In [1]:
# load in libero 
from octo.data.oxe import make_oxe_dataset_kwargs
from octo.data.dataset import make_single_dataset

import tensorflow as tf
import json 

import mediapy
import re


2025-04-06 03:36:12.296483: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-06 03:36:12.301168: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-06 03:36:12.313615: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743910572.333689 3371239 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743910572.339860 3371239 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743910572.356168 3371239 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# load chunk of dataset in based on ID
print("Loading dataset *******")
dataset = 'libero_90'
dataset_kwargs = make_oxe_dataset_kwargs(dataset,"gs://rail-orca-central2/resize_256_256/")
# dataset_kwargs["use_cot"]=True
# dataset_kwargs["cot_data_path"]="/nfs/kun2/users/riadoshi/universal-CoT/final_jsons_copy"
dataset = make_single_dataset(
                                dataset_kwargs, 
                                frame_transform_kwargs=dict(
                                    resize_size={"primary": (256, 256)},
                                ),
                                train=True)
iterator = dataset.iterator()

Loading dataset *******


2025-04-06 03:36:19.272066: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


AttributeError: module 'ml_dtypes' has no attribute 'float4_e2m1fn'
Cause: Unable to locate the source code of <function _gcd_import at 0x7f0544ca7d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f0544ca7d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f0544ca7d80>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-04-06 03:36:23.353787: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


In [3]:
# raw reasoning loader
with tf.io.gfile.GFile('gs://multi-robot-bucket2/scratch/ucot_reasoning_new.json', 'r') as f:
    old_cot_dct = json.load(f)

In [4]:
new_cot_dct = {}
for i, traj in enumerate(iterator):
    traj_id = traj['traj_idx'][0]
    file_path, demo_id = str(traj['metadata']['file_path'][0].decode('utf-8')), str(traj['metadata']['demo_id'][0])

    reasoning_dct = old_cot_dct[f"{file_path}--{demo_id}"]

    # gripper --> end effector
    new_reasoning_dct = reasoning_dct.copy()
    new_reasoning_dct['end_effector_centroids'] = {
        'right_end_effector': reasoning_dct['gripper_centroids'], 
        'left_end_effector': {}
    }
    new_reasoning_dct['obj_masks'] = {}

    # reformat the reasoning dct
    obj_ids = new_reasoning_dct['obj_id_to_name'].keys() # obj_1, obj_2 etc
    traj_len = len(new_reasoning_dct['end_effector_centroids']['right_end_effector'].keys()) # traj len

    # fix obj bboxes
    for obj_id in obj_ids:

        new_reasoning_dct['obj_masks'][f"{obj_id}"] = {}

        for step in range(0, traj_len):
            if f"{obj_id}" in reasoning_dct["obj_bboxes"][f"{step}"]:
                obj_bbox_str = reasoning_dct["obj_bboxes"][f"{step}"][f"{obj_id}"]
                # switch x min and y min, switch x max and y max
                xmin, ymin, xmax, ymax = [match for match in re.findall(r'<loc\d{4}>', obj_bbox_str)]
                obj_bbox_str = f"{ymin}{xmin}{ymax}{xmax}"
            else:
                obj_bbox_str = ""
                
            new_reasoning_dct['obj_masks'][f"{obj_id}"][f"{step}"] = obj_bbox_str

    new_reasoning_dct.pop('gripper_centroids')
    new_reasoning_dct.pop('obj_centroids')
    new_reasoning_dct.pop('obj_bboxes')

    new_cot_dct[f"{traj_id}"] = new_reasoning_dct

    

    if i%500 == 0:
        print(f"processed {i} trajs")


processed 0 trajs


processed 500 trajs
processed 1000 trajs
processed 1500 trajs
processed 2000 trajs
processed 2500 trajs
processed 3000 trajs
processed 3500 trajs


2025-04-06 03:37:41.421903: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [5]:
with open('/nfs/nfs2/users/riadoshi/bigvision_palivla/data_generation/visualization/libero/libero_90_reasonings.json', 'w') as f:
    json.dump(new_cot_dct, f)


In [ ]:
# load in existing file 